# 04 - The worked cases, re-checked

`IDEA.md` 5 tells the story through six specific payment IDs. This notebook
pulls each one out of the regenerated data and checks what the built system
actually does with it.

**There is a problem with using them in the demo, and it is found at the bottom
of this notebook.** Read to the end before building slides on these.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
os.environ['PYTHONUTF8'] = '1'
import numpy as np, warnings
warnings.filterwarnings('ignore')
DATA = '../data300k'      # the working set
DATA100K = '../data'      # the calibrated baseline the pitch quotes

In [2]:
from core.feature_store import FeatureStore
from core.truth import TruthVault
from core.model import Adjudicator
from core.policy import PolicyConfig, decide

# the pitch cases live in the 100k baseline
store, vault = FeatureStore.load(DATA100K), TruthVault(DATA100K)
model = Adjudicator().fit(store, vault)
cfg = PolicyConfig(cap=0.05)
by_id = {e.payment_id: e for e in store.cases}

CASES = [
 ('1  the anniversary gift',      'pay_OTsB19Q7mTYMNh'),
 ('2  clean-looking stolen card', 'pay_W2hA3GA0lkTTCi'),
 ('2b trivially bad',             'pay_UEvFWPHFDVkAbU'),
 ('3  the Kolkata pharmacy order','pay_g8ZOxRE2gbxPX7'),
 ('4  genuinely ambiguous',       'pay_dlTUV3L9E44RuJ'),
 ('5  where it gets it wrong',    'pay_sipFWzV2kBhgMl'),
]
print(f'all six present in the regenerated data: {all(pid in by_id for _, pid in CASES)}')

all six present in the regenerated data: True


Grading these needs three outcomes, not two. Releasing a good order is right;
upholding a bad one is right; **abstaining is not an error** -- it is the
designed response to evidence that cannot settle the question. Only a release
that loses money, or an uphold that refuses a good customer, counts as wrong.

In [3]:
IDEA_EXPECTS = {
 'pay_OTsB19Q7mTYMNh': 'OVERTURN', 'pay_W2hA3GA0lkTTCi': 'UPHOLD',
 'pay_UEvFWPHFDVkAbU': 'UPHOLD',   'pay_g8ZOxRE2gbxPX7': 'OVERTURN',
 'pay_dlTUV3L9E44RuJ': 'abstain',  'pay_sipFWzV2kBhgMl': 'OVERTURN (and lose)',
}

def verdict(action, outcome):
    good = outcome == 'clean'
    if action == 'OVERTURN':
        return 'RIGHT - released a good order' if good else 'WRONG - released a bad one'
    if action == 'UPHOLD':
        return 'WRONG - refused a good customer' if good else 'RIGHT - block stands'
    return 'ABSTAINED - not an error, but no money moved'

for label, pid in CASES:
    e, t = by_id[pid], vault.grade([pid])[0]
    p = model.predict_one(e); d = decide(p, e, cfg); n = e.network
    print(f'{label}')
    print(f'   {pid}  {e.merchant}  Rs {e.amount_inr:,.0f}  blocked for {e.meta["block_reason"]}')
    print(f'   network: {n["network_orders_prior"]:.0f} orders / {n["network_merchants_prior"]:.0f} merchants / '
          f'{n["network_tenure_days"]:.0f}d / clean {n["network_clean_rate"]:.3f} / {n["network_disputes_prior"]:.0f} disputes')
    print(f'   p_bad {p:.4f}  ->  {d.action.value:<9}   truth {t.true_outcome} ({t.persona})   split {t.split}')
    print(f'   {verdict(d.action.value, t.true_outcome)}')
    print(f'   IDEA.md narrates: {IDEA_EXPECTS[pid]}')
    print(f'   reasons: {", ".join(d.reasons)}')
    print()

1  the anniversary gift
   pay_OTsB19Q7mTYMNh  Aurum Jewels  Rs 671,235  blocked for geo_risk
   network: 78 orders / 1 merchants / 1027d / clean 1.000 / 0 disputes
   p_bad 0.0087  ->  OVERTURN    truth clean (legit_stable)   split train
   RIGHT - released a good order
   IDEA.md narrates: OVERTURN
   reasons: ev_positive(+160,538), p_bad_under_cap(0.009<0.05)

2  clean-looking stolen card
   pay_W2hA3GA0lkTTCi  Aurum Jewels  Rs 62,296  blocked for thin_file_high_value
   network: 2 orders / 7 merchants / 33d / clean 1.000 / 0 disputes
   p_bad 1.0000  ->  UPHOLD      truth fraud_undisputed (fraudster)   split train
   RIGHT - block stands
   IDEA.md narrates: UPHOLD
   reasons: confidently_bad(p_bad=1.000>0.55), thin_file_but_uphold_needs_no_exculpation, thin_network_file(orders=2<3)

2b trivially bad
   pay_UEvFWPHFDVkAbU  Aurum Jewels  Rs 614,508  blocked for instrument_risk
   network: 6 orders / 2 merchants / 315d / clean 0.000 / 4 disputes
   p_bad 0.7344  ->  UPHOLD      truth

### Where the built system departs from the written narrative

Three of the six do not play out the way `IDEA.md` tells it. That is worth
knowing before the story is on a slide.

- **Case 3** (the Kolkata pharmacy order) is narrated as an overturn. The model
  scores it around 0.22, and at a Rs 14,627 basket the EV term is negative, so
  it abstains instead. An 88-order, 1,132-day, spotless file *should* clear this
  and does not -- that is a genuine model miss, and the most interesting bug in
  the system right now.
- **Case 4** abstains, exactly as narrated. This one holds.
- **Case 5** is narrated as the expensive mistake, the Rs 8.4L friendly-fraud
  release. At this operating point the system abstains instead, so the demo does
  not get its cautionary tale for free. The costliest *actual* wrong releases are
  in `METRICS.md` 6 -- use one of those.

## Case 2 is the one that earns trust

A perfect 1.000 clean rate, and the system still upholds the block. Two orders
spread across *seven different merchants* in 33 days is not a shopping
pattern -- it is reconnaissance, a stolen instrument being tested thinly and
widely before the real spend.

A model keyed on `clean_rate` releases this. A model that reads the *shape* --
orders against merchant breadth against tenure -- does not. Compare the
orders-per-merchant ratio of case 2 against case 1.

In [4]:
for label, pid in CASES:
    n = by_id[pid].network
    ratio = n['network_orders_prior'] / max(n['network_merchants_prior'], 1)
    per_month = n['network_orders_prior'] / max(n['network_tenure_days']/30, 1)
    print(f'{label:<32} orders/merchant {ratio:>6.1f}   orders/month {per_month:>5.1f}   clean {n["network_clean_rate"]:.3f}')

1  the anniversary gift          orders/merchant   78.0   orders/month   2.3   clean 1.000
2  clean-looking stolen card     orders/merchant    0.3   orders/month   1.8   clean 1.000
2b trivially bad                 orders/merchant    3.0   orders/month   0.6   clean 0.000
3  the Kolkata pharmacy order    orders/merchant   29.3   orders/month   2.3   clean 1.000
4  genuinely ambiguous           orders/merchant    0.5   orders/month   0.2   clean 1.000
5  where it gets it wrong        orders/merchant   16.7   orders/month   1.6   clean 0.900


## Case 5 belongs in the demo precisely because it is a failure

Friendly fraud is committed by real customers with real histories. The evidence
that exonerates an honest atypical buyer looks identical to the evidence
protecting a first-party abuser, because it *is* the same evidence. No
threshold removes this class of error -- the operating point only prices it.

## The problem with these six cases

Five of the six are in the **train** split.

In [5]:
from collections import Counter
splits = Counter(vault.grade([pid])[0].split for _, pid in CASES)
print('split of the six pitch cases:', dict(splits))
print()
print('The model was fitted on the first 80% of train. Showing a live')
print('"RECLAIMIFY overturns this case" on a case the model trained on is')
print('circular, and a judge who asks "was this in your training set?"')
print('gets an answer you do not want to give on camera.')

split of the six pitch cases: {'train': 5, 'holdout': 1}

The model was fitted on the first 80% of train. Showing a live
"RECLAIMIFY overturns this case" on a case the model trained on is
circular, and a judge who asks "was this in your training set?"
gets an answer you do not want to give on camera.


### Holdout substitutes for each narrative role

Same stories, cases the model has never seen. These are the ones to demo.

In [6]:
ho = store.split('holdout')
p_ho = model.predict(store, ho)
truth = {t.payment_id: t for t in vault.grade(store.payment_ids(ho))}

def pick(pred, key, n=2):
    rows = [(e, p) for e, p in zip(ho, p_ho) if pred(e, p, truth[e.payment_id])]
    rows.sort(key=key)
    return rows[:n]

roles = [
 ('long clean file, wrongly blocked, high value (Case 1 role)',
  lambda e,p,t: t.true_outcome=='clean' and e.network['network_orders_prior']>=40
                and e.network['network_clean_rate']>=0.95 and e.amount_inr>100_000,
  lambda r: -r[0].amount_inr),
 ('clean-looking but genuinely bad (Case 2 role)',
  lambda e,p,t: t.true_outcome!='clean' and e.network['network_clean_rate']>=0.95
                and e.network['network_orders_prior']<=6,
  lambda r: -r[0].amount_inr),
 ('thin file, high value -> must abstain (Case 4 role)',
  lambda e,p,t: e.network['network_orders_prior']<3 and e.amount_inr>100_000,
  lambda r: -r[0].amount_inr),
 ('long real history, still goes bad (Case 5 role)',
  lambda e,p,t: t.true_outcome=='chargeback_friendly' and e.network['network_orders_prior']>=20,
  lambda r: -r[0].amount_inr),
]

for name, pred, key in roles:
    print(f'--- {name} ---')
    for e, p in pick(pred, key):
        t = truth[e.payment_id]; d = decide(p, e, cfg); n = e.network
        print(f'  {e.payment_id}  {e.merchant:<14} Rs {e.amount_inr:>11,.0f}  {e.meta["block_reason"]:<20}')
        print(f'     {n["network_orders_prior"]:.0f} orders/{n["network_merchants_prior"]:.0f} merchants/'
              f'{n["network_tenure_days"]:.0f}d/clean {n["network_clean_rate"]:.2f}  ->  p_bad {p:.3f}  {d.action.value}   truth={t.true_outcome}')
    print()

--- long clean file, wrongly blocked, high value (Case 1 role) ---
  pay_ChjlxRbqXfwgAC  Aurum Jewels   Rs     405,894  device_reputation   
     175 orders/1 merchants/1599d/clean 0.96  ->  p_bad 0.009  OVERTURN   truth=clean
  pay_KzFRSz0D4PUwDZ  Aurum Jewels   Rs     378,814  amount_anomaly      
     53 orders/1 merchants/1322d/clean 0.96  ->  p_bad 0.500  STEP_UP   truth=clean

--- clean-looking but genuinely bad (Case 2 role) ---

--- thin file, high value -> must abstain (Case 4 role) ---
  pay_OsnsOd4R0U7Rjb  Aurum Jewels   Rs     234,315  amount_anomaly      
     1 orders/1 merchants/353d/clean 0.00  ->  p_bad 0.913  UPHOLD   truth=chargeback_fraud
  pay_9JowhF6i0ELAHV  Aurum Jewels   Rs     218,820  thin_file_high_value
     2 orders/3 merchants/1195d/clean 1.00  ->  p_bad 0.009  STEP_UP   truth=clean

--- long real history, still goes bad (Case 5 role) ---
  pay_coHRRmPrqtCwHV  SnackBox       Rs       2,564  rto_history         
     21 orders/2 merchants/419d/clean 0.86  -

Swap these into the demo script and the whole run is on data the model has
never seen. It costs an afternoon of slide edits and removes the single
easiest question a judge can ask.